# Preliminary Experiments

SAE stability project — scratch space for preliminary experiments.

In [ ]:
# Install dependencies (run this cell first in Colab)
# !pip install transformer_lens sae_lens datasets torch

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from typing import Tuple, List, Dict
from scipy.optimize import linear_sum_assignment
from transformer_lens import HookedTransformer
from datasets import load_dataset

In [ ]:
# Configuration
CONFIG = {
    "model_name": "pythia-70m-deduped",
    "hook_point": "blocks.3.hook_resid_post",  # Middle layer of Pythia-70m (6 layers, so layer 3)
    "d_model": 512,  # Pythia-70m hidden dimension
    "n_features": 2048,  # SAE dictionary size (4x expansion)
    "l1_coeff": 1e-3,  # Sparsity coefficient
    "lr": 1e-3,
    "batch_size": 256,
    "n_tokens": 1_000_000,  # Tokens for training (reduced for preliminary experiment)
    "seq_len": 128,
    "seeds": [42, 137],  # Two different random seeds
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"Using device: {CONFIG['device']}")

## 1. Load Model and Collect Activations

In [ ]:
# Load the Pythia model
model = HookedTransformer.from_pretrained(
    CONFIG["model_name"],
    device=CONFIG["device"],
)
model.eval()
print(f"Loaded {CONFIG['model_name']} with {model.cfg.n_layers} layers")

In [ ]:
# Load dataset and tokenize
dataset = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)

def collect_activations(model, dataset, hook_point: str, n_tokens: int, seq_len: int, batch_size: int) -> torch.Tensor:
    """Collect activations from the model at the specified hook point."""
    activations_list = []
    tokens_collected = 0
    token_buffer = []
    
    print(f"Collecting {n_tokens:,} tokens of activations from {hook_point}...")
    
    for example in tqdm(dataset, total=n_tokens // seq_len, desc="Collecting activations"):
        # Tokenize and add to buffer
        tokens = model.tokenizer(example["text"], return_tensors="pt", truncation=True, max_length=seq_len * 10)["input_ids"][0]
        token_buffer.extend(tokens.tolist())
        
        # Process when we have enough tokens
        while len(token_buffer) >= seq_len * batch_size:
            # Extract batch
            batch_tokens = torch.tensor(token_buffer[:seq_len * batch_size]).reshape(batch_size, seq_len)
            token_buffer = token_buffer[seq_len * batch_size:]
            
            # Get activations
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    batch_tokens.to(CONFIG["device"]),
                    names_filter=[hook_point],
                )
                acts = cache[hook_point]  # (batch, seq, d_model)
                activations_list.append(acts.reshape(-1, acts.shape[-1]).cpu())
            
            tokens_collected += batch_size * seq_len
            if tokens_collected >= n_tokens:
                break
        
        if tokens_collected >= n_tokens:
            break
    
    activations = torch.cat(activations_list, dim=0)[:n_tokens]
    print(f"Collected activations shape: {activations.shape}")
    return activations

In [ ]:
# Collect activations (this may take a while)
activations = collect_activations(
    model,
    dataset,
    CONFIG["hook_point"],
    CONFIG["n_tokens"],
    CONFIG["seq_len"],
    CONFIG["batch_size"],
)

## 2. Define SAE Architecture

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Standard Sparse Autoencoder with ReLU activation and L1 sparsity penalty.
    
    Architecture:
        encoder: x -> ReLU(W_enc @ (x - b_dec) + b_enc)
        decoder: f -> W_dec @ f + b_dec
    """
    
    def __init__(self, d_model: int, n_features: int, seed: int):
        super().__init__()
        torch.manual_seed(seed)
        
        self.d_model = d_model
        self.n_features = n_features
        
        # Encoder weights and bias
        self.W_enc = nn.Parameter(torch.randn(d_model, n_features) * 0.01)
        self.b_enc = nn.Parameter(torch.zeros(n_features))
        
        # Decoder weights and bias
        self.W_dec = nn.Parameter(torch.randn(n_features, d_model) * 0.01)
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        
        # Initialize decoder columns to unit norm
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=1)
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input to sparse feature activations."""
        x_centered = x - self.b_dec
        pre_acts = x_centered @ self.W_enc + self.b_enc
        return F.relu(pre_acts)
    
    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """Decode feature activations back to input space."""
        return f @ self.W_dec + self.b_dec
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass returning reconstruction, features, and loss components.
        
        Returns:
            x_hat: Reconstructed input
            f: Feature activations
            loss_dict: Dictionary with reconstruction and sparsity losses
        """
        f = self.encode(x)
        x_hat = self.decode(f)
        
        # Reconstruction loss (MSE)
        recon_loss = F.mse_loss(x_hat, x)
        
        # Sparsity loss (L1 on feature activations)
        sparsity_loss = f.abs().mean()
        
        return x_hat, f, {"recon_loss": recon_loss, "sparsity_loss": sparsity_loss}
    
    def normalize_decoder(self):
        """Normalize decoder columns to unit norm (call after each optimization step)."""
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=1)

## 3. Train SAEs with Different Seeds

In [ ]:
def train_sae(
    activations: torch.Tensor,
    d_model: int,
    n_features: int,
    seed: int,
    l1_coeff: float,
    lr: float,
    batch_size: int,
    n_epochs: int = 1,
    device: str = "cuda",
) -> SparseAutoencoder:
    """Train a Sparse Autoencoder on the given activations."""
    
    print(f"\nTraining SAE with seed {seed}...")
    
    # Create SAE
    sae = SparseAutoencoder(d_model, n_features, seed).to(device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)
    
    # Create dataloader
    dataset = TensorDataset(activations)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Training loop
    losses = {"recon": [], "sparsity": [], "total": []}
    
    for epoch in range(n_epochs):
        epoch_losses = {"recon": 0, "sparsity": 0, "total": 0}
        n_batches = 0
        
        for (batch,) in tqdm(dataloader, desc=f"Epoch {epoch + 1}/{n_epochs}"):
            batch = batch.to(device)
            
            # Forward pass
            x_hat, f, loss_dict = sae(batch)
            
            # Compute total loss
            total_loss = loss_dict["recon_loss"] + l1_coeff * loss_dict["sparsity_loss"]
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            # Normalize decoder
            sae.normalize_decoder()
            
            # Track losses
            epoch_losses["recon"] += loss_dict["recon_loss"].item()
            epoch_losses["sparsity"] += loss_dict["sparsity_loss"].item()
            epoch_losses["total"] += total_loss.item()
            n_batches += 1
        
        # Average losses
        for k in epoch_losses:
            epoch_losses[k] /= n_batches
            losses[k].append(epoch_losses[k])
        
        print(f"  Recon: {epoch_losses['recon']:.6f}, Sparsity: {epoch_losses['sparsity']:.6f}, Total: {epoch_losses['total']:.6f}")
    
    return sae, losses

In [ ]:
# Train SAEs with different seeds
trained_saes = {}
training_losses = {}

for seed in CONFIG["seeds"]:
    sae, losses = train_sae(
        activations,
        CONFIG["d_model"],
        CONFIG["n_features"],
        seed,
        CONFIG["l1_coeff"],
        CONFIG["lr"],
        CONFIG["batch_size"],
        n_epochs=3,  # Increase for better training
        device=CONFIG["device"],
    )
    trained_saes[seed] = sae
    training_losses[seed] = losses

print(f"\nTrained {len(trained_saes)} SAEs with seeds: {list(trained_saes.keys())}")

## 4. Feature Matching Between SAEs

In [ ]:
def compute_decoder_similarity(sae1: SparseAutoencoder, sae2: SparseAutoencoder) -> torch.Tensor:
    """
    Compute cosine similarity between decoder columns of two SAEs.
    
    Returns:
        similarity_matrix: (n_features, n_features) matrix where entry (i, j) is
                          the cosine similarity between feature i of sae1 and feature j of sae2
    """
    # Get decoder weights (n_features, d_model)
    W1 = sae1.W_dec.detach()
    W2 = sae2.W_dec.detach()
    
    # Normalize (should already be normalized, but ensure)
    W1_norm = F.normalize(W1, dim=1)
    W2_norm = F.normalize(W2, dim=1)
    
    # Compute cosine similarity matrix
    similarity = W1_norm @ W2_norm.T  # (n_features, n_features)
    
    return similarity.cpu()


def match_features_hungarian(similarity_matrix: torch.Tensor) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Use Hungarian algorithm to find optimal 1-to-1 matching between features.
    
    Returns:
        row_indices: Indices of features in SAE1
        col_indices: Matched indices of features in SAE2
        match_similarities: Cosine similarity for each matched pair
    """
    # Convert to cost matrix (negative similarity for minimization)
    cost_matrix = -similarity_matrix.numpy()
    
    # Solve assignment problem
    row_indices, col_indices = linear_sum_assignment(cost_matrix)
    
    # Get similarities for matched pairs
    match_similarities = similarity_matrix[row_indices, col_indices].numpy()
    
    return row_indices, col_indices, match_similarities


def compute_reappearance_probability(
    saes: Dict[int, SparseAutoencoder],
    similarity_threshold: float = 0.9,
) -> Tuple[np.ndarray, Dict]:
    """
    Compute reappearance probability for each feature in the first SAE.
    
    For 2 SAEs, this is simply whether the feature reappears (0 or 1).
    For N > 2 SAEs, this would be the fraction of other SAEs where the feature reappears.
    
    Args:
        saes: Dictionary mapping seed to trained SAE
        similarity_threshold: Minimum cosine similarity to consider a feature as "reappeared"
    
    Returns:
        reappearance_probs: Array of reappearance probabilities for each feature
        matching_info: Dictionary with detailed matching information
    """
    seeds = list(saes.keys())
    reference_seed = seeds[0]
    reference_sae = saes[reference_seed]
    n_features = reference_sae.n_features
    
    # Track matches for each feature
    reappearance_counts = np.zeros(n_features)
    matching_info = {"matches": [], "similarities": []}
    
    # Compare reference SAE to all other SAEs
    for other_seed in seeds[1:]:
        other_sae = saes[other_seed]
        
        # Compute similarity matrix
        sim_matrix = compute_decoder_similarity(reference_sae, other_sae)
        
        # Find optimal matching
        row_idx, col_idx, match_sims = match_features_hungarian(sim_matrix)
        
        # Store matching info
        matching_info["matches"].append((row_idx, col_idx))
        matching_info["similarities"].append(match_sims)
        
        # Count reappearances (matches above threshold)
        reappearance_counts += (match_sims >= similarity_threshold).astype(float)
    
    # Compute reappearance probability (fraction of other SAEs where feature reappeared)
    n_comparisons = len(seeds) - 1
    reappearance_probs = reappearance_counts / n_comparisons
    
    return reappearance_probs, matching_info

In [ ]:
# Compute reappearance probabilities
SIMILARITY_THRESHOLD = 0.9  # Features with cosine similarity >= 0.9 are considered "reappeared"

reappearance_probs, matching_info = compute_reappearance_probability(
    trained_saes,
    similarity_threshold=SIMILARITY_THRESHOLD,
)

print(f"Computed reappearance probabilities for {len(reappearance_probs)} features")
print(f"Mean reappearance probability: {reappearance_probs.mean():.3f}")
print(f"Median reappearance probability: {np.median(reappearance_probs):.3f}")

## 5. Group Stable and Unstable Features

In [ ]:
# Group features by stability (threshold = 0.5)
STABILITY_THRESHOLD = 0.5

stable_mask = reappearance_probs >= STABILITY_THRESHOLD
unstable_mask = ~stable_mask

stable_indices = np.where(stable_mask)[0]
unstable_indices = np.where(unstable_mask)[0]

print(f"Stability threshold: {STABILITY_THRESHOLD}")
print(f"Stable features: {len(stable_indices)} ({100 * len(stable_indices) / len(reappearance_probs):.1f}%)")
print(f"Unstable features: {len(unstable_indices)} ({100 * len(unstable_indices) / len(reappearance_probs):.1f}%)")

## 6. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Distribution of match similarities
ax1 = axes[0]
similarities = matching_info["similarities"][0]  # Similarities from the one comparison we did
ax1.hist(similarities, bins=50, edgecolor="black", alpha=0.7)
ax1.axvline(SIMILARITY_THRESHOLD, color="red", linestyle="--", label=f"Threshold = {SIMILARITY_THRESHOLD}")
ax1.set_xlabel("Cosine Similarity")
ax1.set_ylabel("Count")
ax1.set_title("Distribution of Feature Match Similarities")
ax1.legend()

# Plot 2: Reappearance probability distribution (simple version)
ax2 = axes[1]
ax2.hist(reappearance_probs, bins=10, edgecolor="black", alpha=0.7)
ax2.axvline(STABILITY_THRESHOLD, color="red", linestyle="--", label=f"Stability threshold = {STABILITY_THRESHOLD}")
ax2.set_xlabel("Reappearance Probability")
ax2.set_ylabel("Count")
ax2.set_title("Distribution of Reappearance Probabilities")
ax2.legend()

# Plot 3: Pie chart of stable vs unstable
ax3 = axes[2]
sizes = [len(stable_indices), len(unstable_indices)]
labels = [f"Stable\n({sizes[0]})", f"Unstable\n({sizes[1]})"]
colors = ["#2ca02c", "#ff7f0e"]
ax3.pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90)
ax3.set_title("Feature Stability Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Create a summary dataframe
import pandas as pd

feature_df = pd.DataFrame({
    "feature_idx": np.arange(len(reappearance_probs)),
    "reappearance_prob": reappearance_probs,
    "match_similarity": matching_info["similarities"][0],
    "is_stable": stable_mask,
})

print("Feature stability summary:")
print(feature_df.describe())

print("\n\nSample of stable features:")
print(feature_df[feature_df["is_stable"]].head(10))

print("\n\nSample of unstable features:")
print(feature_df[~feature_df["is_stable"]].head(10))

## 7. Save Results (Optional)

In [ ]:
# Save results for future use
import os

# Create output directory
os.makedirs("outputs", exist_ok=True)

# Save feature stability data
feature_df.to_csv("outputs/feature_stability.csv", index=False)
print("Saved feature stability data to outputs/feature_stability.csv")

# Save SAE checkpoints
for seed, sae in trained_saes.items():
    torch.save(sae.state_dict(), f"outputs/sae_seed_{seed}.pt")
    print(f"Saved SAE checkpoint to outputs/sae_seed_{seed}.pt")

# Save matching info
np.savez(
    "outputs/matching_info.npz",
    similarities=np.array(matching_info["similarities"]),
    reappearance_probs=reappearance_probs,
    stable_indices=stable_indices,
    unstable_indices=unstable_indices,
)
print("Saved matching info to outputs/matching_info.npz")

## 8. Analyze Stable Features

In [ ]:
# Compare statistics between stable and unstable features
reference_sae = trained_saes[CONFIG["seeds"][0]]

# Get decoder norms
decoder_norms = reference_sae.W_dec.detach().cpu().norm(dim=1).numpy()

# Get activation frequencies (how often each feature fires on our data)
with torch.no_grad():
    all_features = reference_sae.encode(activations.to(CONFIG["device"]))
    activation_freq = (all_features > 0).float().mean(dim=0).cpu().numpy()
    mean_activation = all_features.mean(dim=0).cpu().numpy()

# Compare stable vs unstable
print("=== Stable Features (n={}) ===".format(len(stable_indices)))
print(f"  Decoder norm:      mean={decoder_norms[stable_indices].mean():.3f}, std={decoder_norms[stable_indices].std():.3f}")
print(f"  Activation freq:   mean={activation_freq[stable_indices].mean():.4f}, std={activation_freq[stable_indices].std():.4f}")
print(f"  Mean activation:   mean={mean_activation[stable_indices].mean():.4f}, std={mean_activation[stable_indices].std():.4f}")

print("\n=== Unstable Features (n={}) ===".format(len(unstable_indices)))
print(f"  Decoder norm:      mean={decoder_norms[unstable_indices].mean():.3f}, std={decoder_norms[unstable_indices].std():.3f}")
print(f"  Activation freq:   mean={activation_freq[unstable_indices].mean():.4f}, std={activation_freq[unstable_indices].std():.4f}")
print(f"  Mean activation:   mean={mean_activation[unstable_indices].mean():.4f}, std={mean_activation[unstable_indices].std():.4f}")

In [ ]:
# Visualize: stable vs unstable feature properties
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Decoder norms
ax1 = axes[0]
ax1.hist(decoder_norms[unstable_indices], bins=30, alpha=0.6, label="Unstable", color="#ff7f0e")
ax1.hist(decoder_norms[stable_indices], bins=10, alpha=0.8, label="Stable", color="#2ca02c")
ax1.set_xlabel("Decoder Norm")
ax1.set_ylabel("Count")
ax1.set_title("Decoder Norms: Stable vs Unstable")
ax1.legend()

# Plot 2: Activation frequency
ax2 = axes[1]
ax2.hist(activation_freq[unstable_indices], bins=30, alpha=0.6, label="Unstable", color="#ff7f0e")
ax2.hist(activation_freq[stable_indices], bins=10, alpha=0.8, label="Stable", color="#2ca02c")
ax2.set_xlabel("Activation Frequency")
ax2.set_ylabel("Count")
ax2.set_title("Activation Frequency: Stable vs Unstable")
ax2.legend()

# Plot 3: Scatter - frequency vs decoder norm, colored by stability
ax3 = axes[2]
ax3.scatter(activation_freq[unstable_indices], decoder_norms[unstable_indices], 
            alpha=0.3, label="Unstable", color="#ff7f0e", s=10)
ax3.scatter(activation_freq[stable_indices], decoder_norms[stable_indices], 
            alpha=0.9, label="Stable", color="#2ca02c", s=50, edgecolor="black")
ax3.set_xlabel("Activation Frequency")
ax3.set_ylabel("Decoder Norm")
ax3.set_title("Feature Properties (Stable = Green)")
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Find top activating tokens for each stable feature
# We need to collect some text data with token information

def get_top_activating_examples(
    model, 
    sae, 
    dataset, 
    feature_indices, 
    n_examples=5, 
    n_tokens_to_scan=100_000,
    context_window=10
):
    """Find text examples that most strongly activate each feature."""
    
    results = {idx: [] for idx in feature_indices}
    token_buffer = []
    text_buffer = []
    
    print(f"Scanning {n_tokens_to_scan:,} tokens for top activating examples...")
    
    # Collect tokens with their source text
    tokens_scanned = 0
    for example in tqdm(dataset, total=n_tokens_to_scan // 50):
        text = example["text"]
        tokens = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"][0]
        
        if len(tokens) < 10:
            continue
            
        # Get activations for this sequence
        with torch.no_grad():
            _, cache = model.run_with_cache(
                tokens.unsqueeze(0).to(CONFIG["device"]),
                names_filter=[CONFIG["hook_point"]],
            )
            acts = cache[CONFIG["hook_point"]][0]  # (seq_len, d_model)
            features = sae.encode(acts)  # (seq_len, n_features)
        
        # Check each feature of interest
        for feat_idx in feature_indices:
            feat_acts = features[:, feat_idx].cpu().numpy()
            max_pos = feat_acts.argmax()
            max_val = feat_acts[max_pos]
            
            if max_val > 0:
                # Get context around the max activation
                start = max(0, max_pos - context_window)
                end = min(len(tokens), max_pos + context_window + 1)
                context_tokens = tokens[start:end]
                context_text = model.tokenizer.decode(context_tokens)
                target_token = model.tokenizer.decode(tokens[max_pos])
                
                results[feat_idx].append({
                    "activation": max_val,
                    "context": context_text,
                    "token": target_token,
                    "position": max_pos,
                })
        
        tokens_scanned += len(tokens)
        if tokens_scanned >= n_tokens_to_scan:
            break
    
    # Sort by activation strength and keep top n
    for feat_idx in feature_indices:
        results[feat_idx] = sorted(results[feat_idx], key=lambda x: -x["activation"])[:n_examples]
    
    return results

In [ ]:
# Get top activating examples for stable features
# Need to reload dataset since we exhausted the iterator
dataset = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)

top_examples = get_top_activating_examples(
    model,
    reference_sae,
    dataset,
    stable_indices,
    n_examples=5,
    n_tokens_to_scan=50_000,  # Reduced for speed
)

In [ ]:
# Display top activating examples for each stable feature
print("=" * 80)
print("TOP ACTIVATING EXAMPLES FOR STABLE FEATURES")
print("=" * 80)

for feat_idx in stable_indices:
    examples = top_examples[feat_idx]
    match_sim = matching_info["similarities"][0][feat_idx]
    
    print(f"\n{'='*80}")
    print(f"FEATURE {feat_idx} (match similarity: {match_sim:.3f})")
    print(f"  Activation freq: {activation_freq[feat_idx]:.4f}")
    print(f"  Decoder norm: {decoder_norms[feat_idx]:.3f}")
    print("-" * 80)
    
    if not examples:
        print("  No activating examples found")
        continue
    
    for i, ex in enumerate(examples[:3]):  # Show top 3
        print(f"\n  Example {i+1} (activation: {ex['activation']:.2f}):")
        print(f"    Token: '{ex['token']}'")
        print(f"    Context: ...{ex['context']}...")
    
print("\n" + "=" * 80)

In [ ]:
# PCA visualization of decoder vectors
from sklearn.decomposition import PCA

# Get decoder vectors
decoder_vectors = reference_sae.W_dec.detach().cpu().numpy()

# Fit PCA
pca = PCA(n_components=2)
decoder_2d = pca.fit_transform(decoder_vectors)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

# Plot unstable features
ax.scatter(decoder_2d[unstable_indices, 0], decoder_2d[unstable_indices, 1], 
           alpha=0.3, label="Unstable", color="#ff7f0e", s=15)

# Plot stable features (larger, with labels)
ax.scatter(decoder_2d[stable_indices, 0], decoder_2d[stable_indices, 1], 
           alpha=1.0, label="Stable", color="#2ca02c", s=100, edgecolor="black", linewidth=1.5)

# Label stable features
for idx in stable_indices:
    ax.annotate(str(idx), (decoder_2d[idx, 0], decoder_2d[idx, 1]), 
                fontsize=8, ha="center", va="bottom")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA of Decoder Vectors (Stable Features in Green)")
ax.legend()
plt.tight_layout()
plt.show()